In [ ]:
import os
import re
import pandas as pd
from datetime import datetime
import numpy as np

DATASET_PATH = r"D:\Downloads\ACL19_Release\ACL19_Release"

In [ ]:
all_folders = os.listdir(DATASET_PATH)

target_companies = [
    "American Express", "Facebook", "Ford Motor", "Gap", "Goldman Sachs",
    "Lockheed Martin", "McDonald", "Nvidia", "PepsiCo", "Target",
    "Walt Disney", "Verizon", "PayPal", "Mondelez",
    "Coca-Cola", "AT&T", "JPMorgan", "Walmart"
]

selected_folders = []
for company in target_companies:
    matches = [f for f in all_folders if company.lower() in f.lower()]
    if matches:
        selected_folders.append(matches[0])
    else:
        print(f"WARNING: no match found for {company}")

print(f"Selected {len(selected_folders)} folders")

In [ ]:
def parse_folder(folder_name):
    company_part, date_part = folder_name.rsplit("_", 1)
    call_date = datetime.strptime(date_part, "%Y%m%d").date()
    return {
        "folder_name": folder_name,
        "company": company_part,
        "call_date": call_date,
        "folder_path": os.path.join(DATASET_PATH, folder_name)
    }

manifest_rows = [parse_folder(f) for f in selected_folders]
manifest_df = pd.DataFrame(manifest_rows)
manifest_df

In [ ]:
ticker_map = {
    "American Express Co": "AXP",
    "Facebook, Inc.": "META",
    "Ford Motor": "F",
    "Gap Inc.": "GAP",
    "Goldman Sachs Group": "GS",
    "Lockheed Martin Corp.": "LMT",
    "McDonald's Corp.": "MCD",
    "Nvidia Corporation": "NVDA",
    "PepsiCo Inc.": "PEP",
    "Target Corp.": "TGT",
    "The Walt Disney Company": "DIS",
    "Verizon Communications": "VZ",
    "PayPal": "PYPL",
    "Mondelez International": "MDLZ",
    "Coca-Cola Company (The)": "KO",
    "AT&T Inc.": "T",
    "JPMorgan Chase & Co.": "JPM",
    "Walmart": "WMT"
}

manifest_df["ticker"] = manifest_df["company"].map(ticker_map)
manifest_df

In [ ]:
import yfinance as yf

ticker = "AXP"  # test with one first
test_data = yf.download(ticker, start="2017-02-19", end="2017-06-19")
test_data.head()

In [ ]:
def get_price_window(ticker, call_date, days_before=30, days_after=35):
    start = call_date - pd.Timedelta(days=days_before)
    end = call_date + pd.Timedelta(days=days_after)
    
    data = yf.download(ticker, start=start, end=end, progress=False)
    
    # Flatten MultiIndex columns if present
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    
    return data

# Test on one row
test_row = manifest_df.iloc[0]
test_prices = get_price_window(test_row["ticker"], test_row["call_date"])
test_prices.head()

In [ ]:
def compute_volatility(price_data, call_date, n_days):
    call_date = pd.Timestamp(call_date)
    window = price_data.loc[call_date:call_date + pd.Timedelta(days=n_days)]
    closes = window["Close"]
    
    if len(closes) < 2:
        return np.nan
    
    returns = closes.pct_change().dropna()
    
    if len(returns) < 2: # zero deviation means volatility is undefined (log of zero !! )
        return np.nan
    
    mean_return = returns.mean()
    std_return = np.sqrt(((returns - mean_return) ** 2).sum() / len(returns))
    volatility = np.log(std_return)  # matches paper's eq. 1 exactly
    
    return volatility

v3 = compute_volatility(test_prices, test_row["call_date"], 3)
v7 = compute_volatility(test_prices, test_row["call_date"], 7)
v15 = compute_volatility(test_prices, test_row["call_date"], 15)
v30 = compute_volatility(test_prices, test_row["call_date"], 30)

print(f"v3={v3:.4f}, v7={v7:.4f}, v15={v15:.4f}, v30={v30:.4f}")

In [ ]:
import time

volatility_data = []

for idx, row in manifest_df.iterrows():
    ticker = row["ticker"]
    call_date = row["call_date"]
    
    try:
        prices = get_price_window(ticker, call_date)
        
        v3 = compute_volatility(prices, call_date, 3)
        v7 = compute_volatility(prices, call_date, 7)
        v15 = compute_volatility(prices, call_date, 15)
        v30 = compute_volatility(prices, call_date, 30)
        
        volatility_data.append({
            "v3": v3, "v7": v7, "v15": v15, "v30": v30
        })
        print(f"✓ {ticker}: v3={v3:.3f}, v30={v30:.3f}")
        
    except Exception as e:
        print(f"✗ {ticker} FAILED: {e}")
        volatility_data.append({
            "v3": np.nan, "v7": np.nan, "v15": np.nan, "v30": np.nan
        })
    
    time.sleep(0.5)  # small delay to avoid hammering yfinance's API

# Attach results to manifest
volatility_df = pd.DataFrame(volatility_data)
manifest_df = pd.concat([manifest_df.reset_index(drop=True), volatility_df], axis=1)
manifest_df

**Acoustic feature extraction with librosa**

In [ ]:
import librosa

test_company = manifest_df.iloc[0]
company_folder = test_company["folder_path"]

import glob
mp3_files = glob.glob(os.path.join(company_folder, "**", "*.mp3"), recursive=True)
print(f"Found {len(mp3_files)} mp3 files for {test_company['company']}")
print(mp3_files[0])

y, sr = librosa.load(mp3_files[0], sr=None)
print(f"Audio loaded: {len(y)} samples, sample rate = {sr} Hz, duration = {len(y)/sr:.2f} seconds")

In [ ]:
# Extract pitch (fundamental frequency)
f0, voiced_flag, voiced_probs = librosa.pyin(
    y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'), sr=sr
)

# Extract energy (RMS - root mean square, a measure of loudness)
rms = librosa.feature.rms(y=y)[0]

# Extract tempo-related feature: zero crossing rate (related to speech rate/noisiness)
zcr = librosa.feature.zero_crossing_rate(y)[0]

print(f"Pitch (f0) array shape: {f0.shape}, mean pitch: {np.nanmean(f0):.2f} Hz")
print(f"RMS energy array shape: {rms.shape}, mean energy: {rms.mean():.4f}")
print(f"Zero-crossing rate shape: {zcr.shape}, mean ZCR: {zcr.mean():.4f}")

In [ ]:
def extract_acoustic_features(filepath):
    try:
        y, sr = librosa.load(filepath, sr=None)
        
        if len(y) == 0:
            return None
        
        f0, voiced_flag, voiced_probs = librosa.pyin(
            y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'), sr=sr
        )
        rms = librosa.feature.rms(y=y)[0]
        zcr = librosa.feature.zero_crossing_rate(y)[0]
        
        return {
            "pitch_mean": np.nanmean(f0),
            "pitch_std": np.nanstd(f0),
            "energy_mean": rms.mean(),
            "energy_std": rms.std(),
            "zcr_mean": zcr.mean(),
            "duration": len(y) / sr
        }
    except Exception as e:
        print(f"Failed on {filepath}: {e}")
        return None

# Test on the same file
test_features = extract_acoustic_features(mp3_files[0])
print(test_features)

In [ ]:
def extract_call_features(folder_path):
    mp3_files = glob.glob(os.path.join(folder_path, "**", "*.mp3"), recursive=True)
    
    if len(mp3_files) == 0:
        print(f"No mp3 files found in {folder_path}")
        return None
    
    all_features = []
    for f in mp3_files:
        feat = extract_acoustic_features(f)
        if feat is not None:
            all_features.append(feat)
    
    if len(all_features) == 0:
        return None
    
    features_df = pd.DataFrame(all_features)
    
    # Aggregate across all sentences in the call: mean and std of each per-sentence stat
    call_level = {
        "pitch_mean_avg": features_df["pitch_mean"].mean(),
        "pitch_mean_std": features_df["pitch_mean"].std(),
        "pitch_std_avg": features_df["pitch_std"].mean(),
        "energy_mean_avg": features_df["energy_mean"].mean(),
        "energy_mean_std": features_df["energy_mean"].std(),
        "zcr_mean_avg": features_df["zcr_mean"].mean(),
        "n_sentences": len(all_features),
        "total_duration": features_df["duration"].sum()
    }
    
    return call_level

# Test on AmEx (our first company) — this will take a minute since it's 271 files
test_call = extract_call_features(manifest_df.iloc[0]["folder_path"])
print(test_call)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
import time

call_features_list = []

for idx, row in manifest_df.iterrows():
    start = time.time()
    print(f"Processing {idx+1}/18: {row['company']}...", end=" ")
    
    features = extract_call_features(row["folder_path"])
    
    elapsed = time.time() - start
    
    if features is not None:
        features["company"] = row["company"]
        features["ticker"] = row["ticker"]
        call_features_list.append(features)
        print(f"done in {elapsed:.1f}s ({features['n_sentences']} sentences)")
    else:
        print(f"FAILED ({elapsed:.1f}s)")

acoustic_df = pd.DataFrame(call_features_list)
acoustic_df

In [ ]:
final_df = manifest_df.merge(acoustic_df.drop(columns=["company"]), on="ticker", how="left")
final_df.to_csv("full_dataset.csv", index=False)
print(f"Final dataset: {final_df.shape[0]} rows, {final_df.shape[1]} columns")
final_df.columns.tolist()